In [ ]:

from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

BASE = Path('../../data/supplementary_figures')
PKL_PATH = BASE / '../../data/supplementary_figures/slr_res/DIVA_data/Coast_with_Mang.pkl'
DIVA_PATH = BASE / '../../data/supplementary_figures/slr_res/global_coastal_wetland_model/Input_data/DIVA_database.csv'
OUT_DIR = BASE / '../../figures/supplementary'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = {
    '126': 'SSP1-2.6',
    '245': 'SSP2-4.5',
    '585': 'SSP5-8.5',
}
SCENARIO_NAMES = ['SSP1-2.6', 'SSP2-4.5', 'SSP5-8.5']
SCENARIO_DISPLAY_NAMES = ['Low-warming', 'Intermediate-warming', 'High-warming']
SLR_IDX = 1  # Pop5, consistent with the main manuscript figures.

COLORS = {
    'low_tsm': '#d98f8f',
    'high_tsm': '#8fbea3',
    'non_deltaic': '#9a9a9a',
    'deltaic': '#a69ab8',
}

plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'
plt.rcParams['axes.linewidth'] = 1.0

with PKL_PATH.open('rb') as f:
    data = pickle.load(f)

diva = pd.read_csv(DIVA_PATH)

df = pd.DataFrame({
    'ClsFID': data['ClsFID'].astype(float),
    'area_present_km2': data['area'],
    'agb_present_Mg_per_km2': data['agb_present'],
})
for scenario in SCENARIOS:
    df[f'agb_{scenario}_Mg_per_km2'] = data[f'agb_{scenario}']
    df[f'area_after_SLR_{scenario}_km2'] = data[f'area_after_SLR_{scenario}'][:, SLR_IDX]
    df[f'area_SLR_change_{scenario}_km2'] = df[f'area_after_SLR_{scenario}_km2'] - df['area_present_km2']
    df[f'agb_SLR_change_{scenario}_TgDM'] = (
        df[f'agb_{scenario}_Mg_per_km2'] * df[f'area_SLR_change_{scenario}_km2']
    ) / 1e6

fields = ['CLSFID', 'TSM', 'deltaid', 'mtidalrng', 'wetmang']
df = df.merge(diva[fields].rename(columns={'CLSFID': 'ClsFID'}), on='ClsFID', how='left')

q25 = df['TSM'].quantile(0.25)
q75 = df['TSM'].quantile(0.75)
valid_delta = df['deltaid'].notna() & (df['deltaid'] != -999.999)

groups = {
    'Low TSM': {
        'group_type': 'Sediment availability proxy',
        'mask': df['TSM'] <= q25,
        'color': COLORS['low_tsm'],
        'note': 'bottom 25%',
    },
    'High TSM': {
        'group_type': 'Sediment availability proxy',
        'mask': df['TSM'] >= q75,
        'color': COLORS['high_tsm'],
        'note': 'top 25%',
    },
    'Non-deltaic': {
        'group_type': 'Coastal setting proxy',
        'mask': ~valid_delta,
        'color': COLORS['non_deltaic'],
        'note': 'deltaid missing/-999.999',
    },
    'Deltaic': {
        'group_type': 'Coastal setting proxy',
        'mask': valid_delta,
        'color': COLORS['deltaic'],
        'note': 'deltaid valid',
    },
}

rows = []
for group_name, spec in groups.items():
    sdf = df.loc[spec['mask']].copy()
    area_present = sdf['area_present_km2'].sum()
    agb_present = (sdf['agb_present_Mg_per_km2'] * sdf['area_present_km2']).sum() / 1e6
    for scenario, scenario_label in SCENARIOS.items():
        area_slr_change = sdf[f'area_SLR_change_{scenario}_km2'].sum()
        agb_slr_change = sdf[f'agb_SLR_change_{scenario}_TgDM'].sum()
        rows.append({
            'group_type': spec['group_type'],
            'group': group_name,
            'group_note': spec['note'],
            'scenario': scenario,
            'scenario_label': scenario_label,
            'n_segments': len(sdf),
            'TSM_threshold_q25': q25,
            'TSM_threshold_q75': q75,
            'mean_TSM': sdf['TSM'].mean(),
            'mean_tidal_range_m': sdf['mtidalrng'].mean(),
            'area_present_km2': area_present,
            'area_SLR_change_km2': area_slr_change,
            'area_SLR_change_pct': area_slr_change / area_present * 100,
            'present_AGB_TgDM': agb_present,
            'AGB_SLR_change_TgDM': agb_slr_change,
            'AGB_SLR_change_pct_present_AGB': agb_slr_change / agb_present * 100,
        })

summary = pd.DataFrame(rows)
summary

def plot_grouped_bars(ax, subset, value_col, group_order, colors, panel, title, ylabel=None, ylim=None):
    x = np.arange(len(SCENARIO_NAMES))
    width = 0.34
    offsets = [-width / 2, width / 2]
    for offset, group in zip(offsets, group_order):
        values = (
            subset[subset['group'] == group]
            .set_index('scenario_label')
            .loc[SCENARIO_NAMES, value_col]
            .to_numpy()
        )
        bars = ax.bar(x + offset, values, width=width, color=colors[group], edgecolor='none', label=group)
        for rect, value in zip(bars, values):
            if abs(value) < 2:
                continue
            va = 'top' if value < 0 else 'bottom'
            y = value - 2.0 if value < 0 else value + 2.0
            ax.text(rect.get_x() + rect.get_width() / 2, y, f'{value:.1f}', ha='center', va=va, fontsize=14)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x, SCENARIO_DISPLAY_NAMES, fontsize=16)
    ax.set_title(panel, loc='left', fontweight='bold')
    ax.text(0.02, 0.95, title, transform=ax.transAxes, ha='left', va='top', fontsize=16)
    if ylabel:
        ax.set_ylabel(ylabel)
    if ylim:
        ax.set_ylim(ylim)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='x', labelsize=16)

sediment = summary[summary['group_type'] == 'Sediment availability proxy']
coastal = summary[summary['group_type'] == 'Coastal setting proxy']

fig, axs = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

plot_grouped_bars(
    axs[0, 0], sediment, 'area_SLR_change_pct', ['Low TSM', 'High TSM'],
    {'Low TSM': COLORS['low_tsm'], 'High TSM': COLORS['high_tsm']},
    'a', 'Sediment availability proxy', ylabel='Sea-level-rise-driven area change (%)', ylim=(-90, 15),
)
plot_grouped_bars(
    axs[0, 1], coastal, 'area_SLR_change_pct', ['Non-deltaic', 'Deltaic'],
    {'Non-deltaic': COLORS['non_deltaic'], 'Deltaic': COLORS['deltaic']},
    'b', 'Coastal setting proxy', ylim=(-55, 15),
)
plot_grouped_bars(
    axs[1, 0], sediment, 'AGB_SLR_change_pct_present_AGB', ['Low TSM', 'High TSM'],
    {'Low TSM': COLORS['low_tsm'], 'High TSM': COLORS['high_tsm']},
    'c', 'Sediment availability proxy', ylabel='Sea-level-rise-driven AGB change (%)', ylim=(-105, 15),
)
plot_grouped_bars(
    axs[1, 1], coastal, 'AGB_SLR_change_pct_present_AGB', ['Non-deltaic', 'Deltaic'],
    {'Non-deltaic': COLORS['non_deltaic'], 'Deltaic': COLORS['deltaic']},
    'd', 'Coastal setting proxy', ylim=(-55, 15),
)

for ax in axs[0, :]:
    ax.set_xticklabels([])

fig.set_constrained_layout_pads(w_pad=0.08, h_pad=0.08, wspace=0.08, hspace=0.08)

legend_top = [
    Patch(color=COLORS['low_tsm'], label='Low sediment availability'),
    Patch(color=COLORS['high_tsm'], label='High sediment availability'),
]
legend_bottom = [
    Patch(color=COLORS['non_deltaic'], label='Non-deltaic'),
    Patch(color=COLORS['deltaic'], label='Deltaic'),
]
axs[0, 0].legend(handles=legend_top, loc='lower left', frameon=False, handlelength=0.8, handletextpad=0.3)
axs[0, 1].legend(handles=legend_bottom, loc='lower left', frameon=False, handlelength=0.8, handletextpad=0.3)

fig.savefig(OUT_DIR / 'figS14_sediment_coastal_setting_slr_response.png', dpi=400, bbox_inches='tight')
# Optional PDF export removed for repository-clean reproduction.
plt.show()
